# GATE software week 1 - Shapes and volumes

You will need to look at the GATE software documentation at https://opengate-python.readthedocs.io/en/10.0.2/index.html

#### Import all the packages that you will need for the simulation
1. Opengate is the GATE simulation software.
2. Pyvista helps you to visualise your simulation.
3. NumPy helps is to work with lists of numbers and perform mathematical operations.
4. os. and Pathlib allow us to define the 'path' to the location of files that we want to use in our Google Drive folder.

In [ ]:
import opengate as gate
#!apt-get install -qq xvfb libgl1-mesa-glx
#!pip install pyvista -qq
import pyvista as pv
pv.set_jupyter_backend('static')
pv.global_theme.notebook = True
pv.start_xvfb()
#pv.set_jupyter_backend('html')
import numpy as np
import os
import pathlib
from pathlib import Path

#### Define your units

In [ ]:
m = gate.g4_units.m
cm = gate.g4_units.cm
mm = gate.g4_units.mm
um = gate.g4_units.um
deg = gate.g4_units.deg

#### And your colours
- These are defined by 4 numbers: [RED, GREEN, BLUE, transparency]
- You can only use colours that you define in this list.
- Try to create your own colours and add them to the list :-)

In [ ]:
# Define colours
colours = {
 'cyan': [0, 1, 1, 1],
 'grey': [.7, .7, .7, 0.03],
 'yellow': [1, 1, 0, 1],
 'green': [0, 1, 0, 1], 
 'blue': [0, 0, 1, 1]   
}

## Volumes
Must be defined with:
- type
- name
- size
- mother volume
- position - if no position is specified, volume is positioned at the centre of it's 'mother' volume
- material (from GEANT 4 materials database)
- colour

You first need to initialise your simulation.

In [ ]:
sim = gate.Simulation()
sim.visu = True             # To display the simulation
sim.visu_type = 'vrml'

You can get more information about volume commands by running 'help(sim.volume_manager).  
(It generates a long output so use 'Clear cell output' to hide it!) 

In [ ]:
help(sim.volume_manager)

Define the world as a 2 m x 2 m x 2 m cube (air)  
Available materials can be found at:  
https://geant4-userdoc.web.cern.ch/UsersGuides/ForApplicationDeveloper/html/Appendix/materialNames.html

In [ ]:
world = sim.world
sim.world.size = [5 * m, 5 * m, 5 * m]
sim.world.material = 'G4_AIR'
sim.world.color = colours['grey']

In [ ]:
sim.run(start_new_process=True)  # This allows us to run the simulation multiple times in the same notebook

Create a water-filled cylinder.
- Name: 'cylinder'
- Size: radius 10.8 cm and length 18.6 cm
- Mother volume: 'world' (your box of air!)
- Position: at the centre of the world.
- Material: water
- Colour: cyan (this is light blue!)

In [ ]:
cylinder_vol = sim.add_volume('TubsVolume', 'cylinder')
cylinder_vol.rmin = 0 * cm                                             # This is your inner radius (should be 0 for a 'full cylinder)                                
cylinder_vol.rmax = 10.8 * cm                                          # Outer radius
cylinder_vol.dz = 18.6 * cm / 2.0                                      # dz is half-height (full height is 18.6 cm)               
cylinder_vol.material = 'G4_WATER'
cylinder_vol.color =  colours['cyan']
cylinder_vol.mother = 'world'

In [ ]:
pv.set_jupyter_backend('html')     # This allows you to zoom in and out!
sim.run(start_new_process=True)

Add a plexi-glass (plastic) shell around the outside of the cylinder to contain all the water.  
- Name: 'cylindershell'
- Size: 3.2 mm thick surrounding the water
- Mother volume: 'world'
- Position: at the centre of the world (surrounding the water).
- Material: plexiglass
- Colour: green

In [ ]:
shell = sim.add_volume('TubsVolume', 'cylindershell')
shell.rmin = 10.8 * cm                      # the inner radius (edge of water)
shell.rmax = 11.12 * cm                     # PMMA thickness 3.2 mm --> radius is 10.8 + 0.32 cm
shell.dz = 18.92 * cm / 2.0                 # Length is 18.6 + 0.32 cm = 18.92 cm               
shell.material = 'G4_PLEXIGLASS'
shell.color =  colours['green']
shell.mother = 'world'

Can you run the simulation again?

Can you make another water-filled cylinder, positioned at 50 cm away from 'cylinder_vol?'
- Name: 'cylinder2'
- Size: radius 10.8 cm and length 18.6 cm
- Mother volume: 'world' 
- Position: 50 cm in the 'x' direction
- Material: water
- Colour: cyan  

Shell:
- Size: 3.2 mm thick surrounding the water
- Mother volume: 'world'
- Position: 50 cm in the 'x' direction
- Material: plexiglass
- Colour: green

Can you make another water-filled cylinder, positioned at -50 cm away from the original cylinder in the x-direction and 20 cm in the y-direction?  
Can you rotate it by 90 degrees in the x-direction?  

Rotation matrix:  
- [[1, 0, 0], [0, 0, -1], [0, 1, 0]] 90 degrees in x-direction

- [[0, 0, 1], [0, 1, 0], [-1, 0, 0]] 90 degrees in y-direction

- [[0, -1, 0], [1, 0, 0], [0, 0, 1]] 90 degrees in z-direction

Next we want to import a camera model.. the GE Discovery is already programmmed into GATE so we can import that first and see what it looks like!

In [ ]:
import opengate.contrib.spect.ge_discovery_nm670 as discovery

Gate can't cope with volumes that overlap each other. So we add a 'parallel world' which is the same size, shape and position as the world that the water cylinders live in.

In [ ]:
sim.add_parallel_world('parallel world')

In [ ]:
spect, collimator, crystal = discovery.add_spect_head(sim, 
                                                      name = 'discovery1', 
                                                      collimator_type = False,    # it takes a long time to visualise!!
                                                      rotation_deg = 0,
                                                      crystal_size = '3/8')

spect.mother = 'parallel world'
#discovery.add_digitizer_tc99m(sim, crystal.name, 'digit_tc99m')

#spect = discovery.add_spect_head(sim, "discovery12", collimator_type="lehr")
#crystal = sim.volume_manager.get_volume(f"{spect.name}_crystal")
#discovery.add_digitizer_lu177(sim, crystal.name, "digit_lu177", rotation_deg=15, crystal_size="5/8")

In [ ]:
sim.run(start_new_process=True)

It looks like our SPECT detector is crashing into our water cylinder!  
You can reset the simulation using sim.volume_manager.reset().  
Then you could start again, moving the detector 20 cm away in the y-direction.